In [1]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

In [2]:
print("Loading dataset...")

# Step 1: Use a relative path. 
# '../' moves out of 'notebook' and into the 'ISL' folder.
csv_path = "../dataset/raw/isl_landmarks.csv"

# Step 2: Read the CSV
df = pd.read_csv(csv_path)

y = df["label"]
X_raw = df.drop("label", axis=1)

print(f"Successfully loaded {len(df)} rows.")
df.head()

Loading dataset...
Successfully loaded 26150 rows.


,L_x0,L_y0,L_z0,L_x1,L_y1,L_z1,L_x2,L_y2,L_z2,L_x3,...,R_x18,R_y18,R_z18,R_x19,R_y19,R_z19,R_x20,R_y20,R_z20,label
0,0.754711,0.630020,-5.950000e-07,0.638595,0.526919,-0.022076,0.547095,0.437036,-0.053783,0.485301,...,0.225033,0.762399,-0.121562,0.225785,0.731938,-0.117614,0.200305,0.689587,-0.113799,A
1,0.765359,0.640817,-6.290000e-07,0.648661,0.532778,-0.022108,0.556981,0.439430,-0.048522,0.491174,...,0.229350,0.745687,-0.117332,0.231434,0.716846,-0.111351,0.208115,0.676829,-0.105051,A
2,0.741353,0.624183,-5.510000e-07,0.617331,0.526992,-0.020824,0.526148,0.438756,-0.056323,0.469552,...,0.207501,0.770234,-0.122589,0.206828,0.738522,-0.119324,0.181206,0.697141,-0.115768,A
3,0.733747,0.617151,-5.570000e-07,0.614776,0.528742,-0.022130,0.521997,0.442171,-0.054790,0.463354,...,0.200479,0.769770,-0.116314,0.200930,0.738650,-0.111097,0.175929,0.698244,-0.106023,A
4,0.733290,0.623661,-5.820000e-07,0.612217,0.533343,-0.022231,0.519188,0.446406,-0.054795,0.460401,...,0.190615,0.773850,-0.122199,0.189554,0.739927,-0.117145,0.165191,0.696974,-0.111792,A


In [3]:
print("Converting raw coordinates to relative distances...")

def compute_distances(data):
    features = pd.DataFrame()
    
    # Left Hand Distances
    for i in range(1, 21):
        features[f'L_dist_{i}'] = np.sqrt(
            (data[f'L_x{i}'] - data['L_x0'])**2 +
            (data[f'L_y{i}'] - data['L_y0'])**2 +
            (data[f'L_z{i}'] - data['L_z0'])**2
        )
        
    # Right Hand Distances
    for i in range(1, 21):
        features[f'R_dist_{i}'] = np.sqrt(
            (data[f'R_x{i}'] - data['R_x0'])**2 +
            (data[f'R_y{i}'] - data['R_y0'])**2 +
            (data[f'R_z{i}'] - data['R_z0'])**2
        )
    
    return features.fillna(0)

# Apply the transformation
X_features = compute_distances(X_raw)
print("Transformation complete. New shape:", X_features.shape)

Converting raw coordinates to relative distances...
Transformation complete. New shape: (26150, 40)


In [4]:
encoder = LabelEncoder()

y_encoded = encoder.fit_transform(y)

joblib.dump(encoder, "../models/label_encoder.pkl")
print("Label encoder saved")

Label encoder saved


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X_features, 
    y_encoded, 
    test_size=0.2, 
    stratify=y_encoded, 
    random_state=42
)

In [6]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, "../models/scaler40.pkl")
print("Scaler saved")

Scaler saved


In [7]:
# Initialize models with regularization to prevent overfitting
rf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
knn = KNeighborsClassifier(n_neighbors=7)
mlp = MLPClassifier(hidden_layer_sizes=(128,64), alpha=0.01, max_iter=400, random_state=42)

# probability=True is required for the SVM to participate in "soft" voting
svm = SVC(kernel='rbf', C=1.0, probability=True, random_state=42)

In [8]:
print("Training the Super Ensemble Model. This might take a few minutes...")

ensemble_model = VotingClassifier(
    estimators=[
        ('Random Forest', rf),
        ('KNN', knn),
        ('MLP', mlp),
        ('SVM', svm)
    ],
    voting='soft'
)

ensemble_model.fit(X_train_scaled, y_train)
print("Ensemble training completed!")

Training the Super Ensemble Model. This might take a few minutes...
Ensemble training completed!


In [9]:
y_pred = ensemble_model.predict(X_test_scaled)

print(f"Ensemble Model Accuracy: {accuracy_score(y_test, y_pred):.4f}")

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, target_names=encoder.classes_))

e:\MANUMOTION\islenv\lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "e:\MANUMOTION\islenv\lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
  File "C:\Users\aatif\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 503, in run
    with Popen(*popenargs, **kwargs) as process:
  File "C:\Users\aatif\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 971, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\aatif\AppData\Local\Programs\Python\Python310\lib\subprocess.py", line 1440, in _execute_child
    hp, ht, pid, tid = _win

Ensemble Model Accuracy: 0.9990

Classification Report:

              precision    recall  f1-score   support

           A       1.00      1.00      1.00       203
           B       1.00      1.00      1.00       250
           C       1.00      1.00      1.00       187
           D       1.00      1.00      1.00       241
           E       1.00      1.00      1.00       203
           F       1.00      1.00      1.00       192
           G       1.00      1.00      1.00       178
           H       1.00      1.00      1.00       176
           I       1.00      0.99      1.00       176
           J       1.00      1.00      1.00       168
           K       1.00      1.00      1.00       167
           L       1.00      1.00      1.00       218
           M       1.00      1.00      1.00       249
           N       1.00      1.00      1.00       213
           O       0.98      1.00      0.99       174
           P       1.00      1.00      1.00       177
           Q       1.00 

In [10]:
joblib.dump(ensemble_model, "../models/ensemble_model.pkl")

print("Saved as 'ensemble_model.pkl'")

Saved as 'ensemble_model.pkl'
